In [3]:
import os
import cv2
import re
from paddleocr import PaddleOCR
from tensorflow.keras.models import load_model

In [4]:
# Fungsi untuk memuat model dari file .h5
def load_model_from_h5(model_path):
    # Memuat model dari file .h5
    model = load_model(model_path)
    print(f"Model berhasil dimuat dari {model_path}")
    return model

# Fungsi untuk memulai kamera dan melakukan pemindaian KTP
def scan_ktp_with_model(model):
    cap = cv2.VideoCapture(1)  # Mengakses kamera (0 adalah ID default untuk kamera)

    if not cap.isOpened():
        print("Tidak dapat mengakses kamera!")
        return

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Tidak dapat membaca frame!")
            break

        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR
            print("Scanning gambar...")

            # Proses OCR pada gambar
            ocr_text = ocr_on_ktp(frame)
            print("Teks hasil OCR yang dinormalisasi:", ocr_text)

            # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
            extracted_data = extract_data_from_ocr(ocr_text)

            # Menampilkan data yang diekstraksi
            print(f"Data yang diekstraksi dari OCR:")
            print(f"NIK: {extracted_data['nik']}")
            print(f"Nama: {extracted_data['nama']}")
            print(f"Alamat: {extracted_data['alamat']}")
            print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
            print(f"Kecamatan: {extracted_data['kecamatan']}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")

    # Menutup kamera dan jendela OpenCV
    cap.release()
    cv2.destroyAllWindows()

# Nama file model .h5
model_path = 'model_ocr_ktp.h5'  # Ganti dengan lokasi model .h5 yang sesuai

# Memuat model
model = load_model_from_h5(model_path)

# Mulai pemindaian KTP dengan model yang telah dimuat
scan_ktp_with_model(model)

[2025-01-16 10:34:33,491] [ WARNING] saving_utils.py:256 - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


Model berhasil dimuat dari model_ocr_ktp.h5
Scanning gambar...


NameError: name 'ocr_on_ktp' is not defined

In [11]:
import os
import cv2
import re
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
from paddleocr import PaddleOCR

# Fungsi untuk memuat model dari file .h5
def load_model_from_h5(model_path):
    # Memuat model dari file .h5
    model = load_model(model_path)
    print(f"Model berhasil dimuat dari {model_path}")
    return model

# Fungsi untuk mempersiapkan gambar agar cocok dengan model yang dimuat
def preprocess_image_for_model(image):
    # Misalnya, jika model mengharapkan input gambar berukuran 224x224 dan dengan 3 saluran (RGB)
    image_resized = cv2.resize(image, (224, 224))  # Sesuaikan ukuran input model Anda
    image_resized = cv2.cvtColor(image_resized, cv2.COLOR_BGR2RGB)  # Pastikan gambar dalam format RGB
    image_array = img_to_array(image_resized)  # Mengonversi gambar ke array
    image_array = np.expand_dims(image_array, axis=0)  # Menambah dimensi batch
    image_array = image_array / 255.0  # Normalisasi gambar jika diperlukan
    return image_array

# Fungsi untuk memulai kamera dan melakukan pemindaian KTP
def scan_ktp_with_model(model):
    cap = cv2.VideoCapture(1)  # Mengakses kamera (1 adalah ID kamera kedua, sesuaikan jika perlu)

    if not cap.isOpened():
        print("Tidak dapat mengakses kamera!")
        return

    # Menggunakan PaddleOCR untuk OCR
    ocr = PaddleOCR(use_angle_cls=True, lang='id')  # Sesuaikan dengan bahasa yang diinginkan

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Tidak dapat membaca frame!")
            break

        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR
            print("Scanning gambar...")

            # Preprocessing gambar untuk model
            processed_image = preprocess_image_for_model(frame)

            # Prediksi teks menggunakan model
            try:
                predicted_text = model.predict(processed_image)
                predicted_text = np.argmax(predicted_text, axis=-1)  # Jika model mengeluarkan hasil prediksi kategori

                # Mengkonversi hasil prediksi ke teks
                ocr_text = decode_prediction(predicted_text)

                print("Teks hasil OCR yang dinormalisasi:", ocr_text)

                # Menggunakan PaddleOCR untuk mendapatkan teks dari gambar
                ocr_result = ocr.ocr(frame, cls=True)
                ocr_text = " ".join([line[1] for line in ocr_result[0]])

                print("Teks hasil OCR dari PaddleOCR:", ocr_text)

                # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
                extracted_data = extract_data_from_ocr(ocr_text)

                # Menampilkan data yang diekstraksi
                print(f"Data yang diekstraksi dari OCR:")
                print(f"NIK: {extracted_data['nik']}")
                print(f"Nama: {extracted_data['nama']}")
                print(f"Alamat: {extracted_data['alamat']}")
                print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
                print(f"Kecamatan: {extracted_data['kecamatan']}")

                # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
                image_filename = "scanned_ktp.jpg"
                cv2.imwrite(image_filename, frame)
                print(f"Gambar disimpan di {image_filename}")

            except Exception as e:
                print(f"Error saat melakukan prediksi: {e}")

    # Menutup kamera dan jendela OpenCV
    cap.release()
    cv2.destroyAllWindows()

# Fungsi untuk mendekodekan hasil prediksi menjadi teks (implementasi tergantung pada model)
def decode_prediction(predicted_text):
    # Misalnya, jika model mengeluarkan hasil berupa angka kategori, kita bisa memetakan angka tersebut ke karakter.
    # Implementasikan decoding sesuai dengan cara model Anda dilatih.
    # Contoh sederhana (misal model mengeluarkan angka yang mewakili karakter):
    character_map = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'  # Sesuaikan dengan karakter yang dikenali oleh model
    decoded_text = ''.join([character_map[int(i)] for i in predicted_text])  # Mentransformasikan hasil prediksi menjadi karakter
    return decoded_text

# Fungsi untuk ekstrak data dari teks OCR
def extract_data_from_ocr(ocr_text):
    data = {
        'nik': None,
        'nama': None,
        'alamat': None,
        'kelurahan': None,
        'kecamatan': None
    }

    nik_match = re.search(r'\d{16}', ocr_text)
    if nik_match:
        data['nik'] = nik_match.group()

    nama_match = re.search(r'Nama\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if nama_match:
        data['nama'] = normalize_text(nama_match.group(1))

    alamat_match = re.search(r'Alamat\s*[:\-]?\s*([A-Z0-9\s.,]+)', ocr_text)
    if alamat_match:
        alamat = alamat_match.group(1)
        alamat_cleaned = re.sub(r'\b(RT|RW|RTRW|RT/RW|R\.T|R\.W)\b\s*\d+(\s*\d+)?', '', alamat, flags=re.IGNORECASE)
        data['alamat'] = normalize_text(alamat_cleaned)

    kelurahan_match = re.search(r'(Kelurahan|Desa)\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kelurahan_match:
        data['kelurahan'] = normalize_text(kelurahan_match.group(2))

    kecamatan_match = re.search(r'Kecamatan\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kecamatan_match:
        data['kecamatan'] = normalize_text(kecamatan_match.group(1))

    return data

# Memuat model
model_path = 'model_ocr_ktp.h5'  # Ganti dengan lokasi model .h5 yang sesuai
model = load_model_from_h5(model_path)

# Mulai pemindaian KTP dengan model yang telah dimuat
scan_ktp_with_model(model)

[2025-01-16 11:07:40,778] [ WARNING] saving_utils.py:256 - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


Model berhasil dimuat dari model_ocr_ktp.h5
[2025/01/16 11:07:41] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\user/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\user/.paddleocr/whl\\rec\\latin\\latin_PP-OCRv3_rec_infer', rec_image_inverse=Tr

In [1]:
import os
import cv2
import re
from paddleocr import PaddleOCR
from tensorflow.keras.models import load_model

# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk membersihkan dan menormalisasi teks hasil OCR
def normalize_text(text):
    text = re.sub(r'\s+', ' ', text)  # Menghapus spasi ganda
    text = re.sub(r'\b[A-Z0-9]\b', '', text)  # Menghapus karakter tunggal
    text = re.sub(r'[^\w\s.,]', '', text)  # Menghapus karakter non-alfanumerik kecuali spasi, titik, dan koma
    text = text.strip()  # Menghapus spasi di awal/akhir
    return text

# Fungsi untuk melakukan OCR pada gambar KTP menggunakan PaddleOCR
def ocr_on_ktp(image):
    ocr = PaddleOCR(use_angle_cls=True, lang='id')  # Menggunakan 'id' untuk Bahasa Indonesia
    result = ocr.ocr(image, cls=True)
    if result and result[0]:
        ocr_text = " ".join([line[1][0] for line in result[0]])
    else:
        ocr_text = "Tidak ada teks yang terdeteksi."
    return normalize_text(ocr_text)

# Fungsi untuk ekstrak data dari teks OCR
def extract_data_from_ocr(ocr_text):
    data = {
        'nik': None,
        'nama': None,
        'alamat': None,
        'kelurahan': None,
        'kecamatan': None
    }

    nik_match = re.search(r'\d{16}', ocr_text)
    if nik_match:
        data['nik'] = nik_match.group()

    nama_match = re.search(r'Nama\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if nama_match:
        data['nama'] = normalize_text(nama_match.group(1))

    alamat_match = re.search(r'Alamat\s*[:\-]?\s*([A-Z0-9\s.,]+)', ocr_text)
    if alamat_match:
        alamat = alamat_match.group(1)
        alamat_cleaned = re.sub(r'\b(RT|RW|RTRW|RT/RW|R\.T|R\.W)\b\s*\d+(\s*\d+)?', '', alamat, flags=re.IGNORECASE)
        data['alamat'] = normalize_text(alamat_cleaned)

    kelurahan_match = re.search(r'(Kelurahan|Desa)\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kelurahan_match:
        data['kelurahan'] = normalize_text(kelurahan_match.group(2))

    kecamatan_match = re.search(r'Kecamatan\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kecamatan_match:
        data['kecamatan'] = normalize_text(kecamatan_match.group(1))

    return data

# Fungsi untuk memuat model dari file .h5
def load_model_from_h5(model_path):
    # Memuat model dari file .h5
    model = load_model(model_path)
    print(f"Model berhasil dimuat dari {model_path}")
    return model

# Fungsi untuk memulai kamera dan melakukan pemindaian KTP
def scan_ktp_with_model(model):
    cap = cv2.VideoCapture(1)  # Mengakses kamera (1 adalah ID kamera kedua, sesuaikan jika perlu)

    if not cap.isOpened():
        print("Tidak dapat mengakses kamera!")
        return

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Tidak dapat membaca frame!")
            break

        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR
            print("Scanning gambar...")

            # Proses OCR pada gambar
            ocr_text = ocr_on_ktp(frame)
            print("Teks hasil OCR yang dinormalisasi:", ocr_text)

            # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
            extracted_data = extract_data_from_ocr(ocr_text)

            # Menampilkan data yang diekstraksi
            print(f"Data yang diekstraksi dari OCR:")
            print(f"NIK: {extracted_data['nik']}")
            print(f"Nama: {extracted_data['nama']}")
            print(f"Alamat: {extracted_data['alamat']}")
            print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
            print(f"Kecamatan: {extracted_data['kecamatan']}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")

    # Menutup kamera dan jendela OpenCV
    cap.release()
    cv2.destroyAllWindows()

# Nama file model .h5
model_path = 'model_ocr_ktp.h5'  # Ganti dengan lokasi model .h5 yang sesuai

# Memuat model
model = load_model_from_h5(model_path)

# Mulai pemindaian KTP dengan model yang telah dimuat
scan_ktp_with_model(model)

[2025-01-16 10:47:56,852] [ WARNING] saving_utils.py:256 - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


Model berhasil dimuat dari model_ocr_ktp.h5
Scanning gambar...
[2025/01/16 10:48:35] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\user/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\user/.paddleocr/whl\\rec\\latin\\latin_PP-OCRv3_rec_infer', r

In [1]:
import cv2

# Memuat file Haar Cascade untuk deteksi wajah
# Pastikan Anda memiliki file "haarcascade_frontalface_default.xml" di direktori yang sesuai
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Membuka kamera atau video
video_capture = cv2.VideoCapture(1)  # 0 untuk kamera default, atau ganti dengan path video

if not video_capture.isOpened():
    print("Error: Kamera atau video tidak dapat dibuka.")
    exit()

while True:
    # Membaca frame dari video
    ret, frame = video_capture.read()
    if not ret:
        print("Error: Tidak dapat membaca frame dari video.")
        break

    # Konversi frame ke skala abu-abu (grayscale)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Deteksi wajah pada frame
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Menggambar kotak di sekitar wajah yang terdeteksi
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

    # Menampilkan hasil di jendela
    cv2.imshow('Face Detection', frame)

    # Menekan tombol 'q' untuk keluar
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Melepaskan kamera dan menutup jendela
video_capture.release()
cv2.destroyAllWindows()


In [1]:
import cv2

# Memuat file Haar Cascade untuk deteksi wajah
# Pastikan Anda memiliki file "haarcascade_frontalface_default.xml" di direktori yang sesuai
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Membuka kamera atau video
video_capture = cv2.VideoCapture(1)  # 0 untuk kamera default, atau ganti dengan path video

if not video_capture.isOpened():
    print("Error: Kamera atau video tidak dapat dibuka.")
    exit()

while True:
    # Membaca frame dari video
    ret, frame = video_capture.read()
    if not ret:
        print("Error: Tidak dapat membaca frame dari video.")
        break

    # Konversi frame ke skala abu-abu (grayscale)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Deteksi wajah pada frame
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Menggambar kotak di sekitar wajah yang terdeteksi
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

    # Menampilkan hasil di jendela
    cv2.imshow('Face Detection', frame)

    # Menekan tombol 'q' untuk keluar
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Melepaskan kamera dan menutup jendela
video_capture.release()
cv2.destroyAllWindows()

In [1]:
import cv2

url = 'http://10.50.1.149:8080/video'  # Ganti dengan URL stream DroidCam Anda
cap = cv2.VideoCapture(url)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Gagal membaca frame.")
        break
    cv2.imshow("DroidCam Stream", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):  # Keluar dengan 'q'
        break

cap.release()
cv2.destroyAllWindows()

Gagal membaca frame.
